In [24]:
!pip install imblearn

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

import seaborn as sns

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical

# Load dataset
data = pd.read_csv('/content/NF-UQ-NIDS new.csv')




In [27]:
data.columns

Index(['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES',
       'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'TCP_FLAGS',
       'FLOW_DURATION_MILLISECONDS', 'Label'],
      dtype='object')

In [28]:
#data.rename(columns={'Label_encoded': 'Label'}, inplace=True)

In [29]:
#data.drop(['IPV4_SRC_ADDR','IPV4_DST_ADDR', 'Attack' ],axis=1,inplace=True)

In [30]:
data.describe()

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,OUT_BYTES,IN_PKTS,OUT_PKTS,TCP_FLAGS,FLOW_DURATION_MILLISECONDS,Label
count,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06
mean,3.258924e+04,1.338601e+04,1.032074e+01,1.747348e+01,5.641052e+03,4.204687e+04,3.907041e+01,4.990077e+01,1.890524e+01,7.950891e+02,6.532389e-02
std,1.932496e+04,1.942791e+04,1.508692e+01,2.125647e+01,9.520146e+04,1.785411e+05,9.763557e+01,1.340718e+02,1.204489e+01,1.380824e+04,2.470966e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.600000e+01,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.572900e+04,5.300000e+01,6.000000e+00,5.000000e+00,5.200000e+02,3.040000e+02,4.000000e+00,4.000000e+00,0.000000e+00,4.000000e+00,0.000000e+00
50%,3.265100e+04,1.344000e+03,6.000000e+00,7.000000e+00,1.684000e+03,3.080000e+03,1.600000e+01,1.800000e+01,2.700000e+01,2.900000e+01,0.000000e+00
75%,4.944800e+04,2.307650e+04,1.700000e+01,3.600000e+01,3.728000e+03,1.944000e+04,4.800000e+01,4.600000e+01,2.700000e+01,3.970000e+02,0.000000e+00
max,6.553500e+04,6.553500e+04,2.550000e+02,2.490000e+02,2.685425e+07,1.465675e+07,2.003800e+04,1.102400e+04,3.100000e+01,3.784558e+06,1.000000e+00


In [31]:
data

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,OUT_BYTES,IN_PKTS,OUT_PKTS,TCP_FLAGS,FLOW_DURATION_MILLISECONDS,Label
0,62073,56082,6,0.0,9672,416,11,8,25,15,0
1,32284,1526,6,0.0,1776,104,6,2,25,0,0
2,21,21971,6,1.0,1842,1236,26,22,25,1111,0
3,23800,46893,6,0.0,528,8824,10,12,27,124,0
4,63062,21,6,1.0,1786,2340,32,34,25,1459,0
...,...,...,...,...,...,...,...,...,...,...,...
1048570,33823,63856,6,36.0,2854,30654,46,48,27,74,0
1048571,37659,35420,6,36.0,2646,24004,42,44,27,52,0
1048572,15965,53,17,5.0,146,178,2,2,0,1,0
1048573,63845,25,6,3.0,37302,3380,52,42,27,24,0


In [32]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import KMeansSMOTE
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans  # Import KMeans for custom estimator

# Function to split data into training and test sets
def simple_split(data):
    train_data, test_data = train_test_split(data, test_size=0.3, random_state=42)
    return train_data, test_data

# Splitting the dataset
train_data, test_data = simple_split(data)

# Separating features (X) and target (y)
X_train = train_data.drop(columns=['Label'])
y_train = train_data['Label']

# Standardizing features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Applying K-Means SMOTE to handle class imbalance
# Use kmeans_estimator to specify the desired number of clusters
kmeans_smote = KMeansSMOTE(random_state=42, k_neighbors=2,
                           cluster_balance_threshold=0.1,
                           kmeans_estimator=KMeans(n_clusters=10))  # Specify n_clusters here
X_resampled, y_resampled = kmeans_smote.fit_resample(X_train_scaled, y_train)

# Now `X_resampled` and `y_resampled` contain the oversampled data
# Convert resampled data back to a DataFrame
X_resampled_df = pd.DataFrame(X_resampled, columns=X_train.columns)
y_resampled_df = pd.DataFrame(y_resampled, columns=['Label'])

# Combine features and target into a single DataFrame
train_data = pd.concat([X_resampled_df, y_resampled_df], axis=1)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The KMeansSMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


In [33]:
train_data

,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,OUT_BYTES,IN_PKTS,OUT_PKTS,TCP_FLAGS,FLOW_DURATION_MILLISECONDS,Label
0,-0.870268,-0.687320,0.444764,-0.587832,-0.056472,-0.235241,-0.375982,-0.357599,-1.572483,-0.063947,0
1,0.689293,-0.685932,-0.286194,-0.493629,-0.040659,-0.179421,-0.254495,-0.238529,0.670936,0.017644,0
2,1.074008,0.881327,-0.286194,-0.305222,-0.019889,-0.222513,-0.213999,-0.238529,0.670936,-0.063622,0
3,-1.035405,-0.685932,-0.286194,-0.493629,-0.040659,-0.179421,-0.254495,-0.238529,0.670936,0.017969,0
4,1.055999,-0.336241,-0.286194,0.919421,0.077150,2.826948,1.912026,2.887061,0.670936,0.072362,0
...,...,...,...,...,...,...,...,...,...,...,...
1372231,-1.685860,-0.690046,13.070396,-0.823340,-0.055917,-0.236236,-0.375982,-0.372483,-1.572483,-0.064028,1
1372232,-1.685860,-0.690046,10.678171,-0.823340,-0.055917,-0.236236,-0.375982,-0.372483,-1.572483,-0.064028,1
1372233,1.690361,-0.688760,-0.286194,-0.682035,0.261794,-0.224701,0.012847,-0.182022,0.670936,0.191711,1
1372234,-1.685860,-0.690046,-0.618447,2.991895,-0.056657,-0.236236,-0.365858,-0.372483,-1.572483,307.183591,1


In [34]:


# Remove rows that contain NaN values
train_data = train_data.dropna()

# Display the cleaned DataFrame
print(train_data)


         L4_SRC_PORT  L4_DST_PORT   PROTOCOL  L7_PROTO  IN_BYTES  OUT_BYTES  \
0          -0.870268    -0.687320   0.444764 -0.587832 -0.056472  -0.235241   
1           0.689293    -0.685932  -0.286194 -0.493629 -0.040659  -0.179421   
2           1.074008     0.881327  -0.286194 -0.305222 -0.019889  -0.222513   
3          -1.035405    -0.685932  -0.286194 -0.493629 -0.040659  -0.179421   
4           1.055999    -0.336241  -0.286194  0.919421  0.077150   2.826948   
...              ...          ...        ...       ...       ...        ...   
1372231    -1.685860    -0.690046  13.070396 -0.823340 -0.055917  -0.236236   
1372232    -1.685860    -0.690046  10.678171 -0.823340 -0.055917  -0.236236   
1372233     1.690361    -0.688760  -0.286194 -0.682035  0.261794  -0.224701   
1372234    -1.685860    -0.690046  -0.618447  2.991895 -0.056657  -0.236236   
1372235    -0.107277    -0.667268  -0.286194  3.462911 -0.043608  -0.212267   

          IN_PKTS  OUT_PKTS  TCP_FLAGS  FLOW_DURATI

In [35]:
# Remove rows that contain NaN values
test_data = test_data.dropna()

# Display the cleaned DataFrame
print(test_data)

        L4_SRC_PORT  L4_DST_PORT  PROTOCOL  L7_PROTO  IN_BYTES  OUT_BYTES  \
781974        29424        26482        17      11.0       544        304   
937737        56798        28420         6      36.0      2854      31354   
907828        56772           53        17       5.0       146        178   
784628        38333        38974         6      11.0      4976       3080   
662460        30455        58354         6      36.0      5486      92132   
...             ...          ...       ...       ...       ...        ...   
49012         54530        53682         6      36.0      3614      47722   
468748         1708           22         6      92.0      3728       5474   
105662         3728           53        17       5.0       146        178   
165822         4156           53        17       5.0       130        162   
681610        10609           22         6      92.0      3728       5474   

        IN_PKTS  OUT_PKTS  TCP_FLAGS  FLOW_DURATION_MILLISECONDS  Label  
7

In [36]:
train_data['Label'].value_counts()

,count
Label,
1,686120
0,686116


Echo State Network

In [37]:

# Manual Echo State Network Implementation
class ManualESN:
    def __init__(self, input_dim, reservoir_size, spectral_radius=0.95, sparsity=0.1):
        self.input_dim = input_dim
        self.reservoir_size = reservoir_size
        self.spectral_radius = spectral_radius
        self.sparsity = sparsity
        self.W_in = np.random.uniform(-1, 1, (reservoir_size, input_dim))
        self.W = np.random.uniform(-1, 1, (reservoir_size, reservoir_size))
        self.W[np.random.rand(*self.W.shape) > sparsity] = 0
        max_eig = max(abs(np.linalg.eigvals(self.W)))
        self.W *= spectral_radius / max_eig
        self.state = np.zeros(reservoir_size)

    def update(self, input_data):
        # Convert input_data to a NumPy array with float64 dtype
        input_data = np.array(input_data, dtype=np.float64)
        self.state = np.tanh(np.dot(self.W_in, input_data) + np.dot(self.W, self.state))
        return self.state

    def fit_transform(self, X):
        states = []
        for x in X:
            states.append(self.update(x))
        return np.array(states)


# DNN Training
class ESN_DNN:
    def __init__(self, reservoir_size=200, spectral_radius=0.95):
        self.esn = ManualESN(input_dim=train_data.drop('Label', axis=1).shape[1],
                             reservoir_size=reservoir_size, spectral_radius=spectral_radius)
        self.dnn = None

    def train(self, X, y):
        esn_output = self.esn.fit_transform(X)
        self.dnn = Sequential([
            Dense(64, activation='relu', input_shape=(esn_output.shape[1],)),
            Dropout(0.15),
            Dense(64, activation='relu'),
            Dropout(0.15),
            Dense(128, activation='relu'),
            Dropout(0.25),
            Dense(32, activation='relu'),
            Dense(len(np.unique(y)), activation='softmax')
        ])
        self.dnn.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        self.dnn.fit(esn_output, y, epochs=15, batch_size=64, verbose=1)

    def predict(self, X):
        esn_output = self.esn.fit_transform(X)
        return self.dnn.predict(esn_output)

# Train and evaluate the model
esn_dnn_model = ESN_DNN(reservoir_size=200, spectral_radius=0.95)
esn_dnn_model.train(train_data.drop('Label', axis=1).values, train_data['Label'].values)




/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 84s 4ms/step - accuracy: 0.9741 - loss: 0.0757
Epoch 2/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 143s 4ms/step - accuracy: 0.9794 - loss: 0.0592
Epoch 3/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 141s 4ms/step - accuracy: 0.9804 - loss: 0.0568
Epoch 4/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 144s 4ms/step - accuracy: 0.9809 - loss: 0.0571
Epoch 5/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 140s 4ms/step - accuracy: 0.9807 - loss: 0.0580
Epoch 6/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 81s 4ms/step - accuracy: 0.9806 - loss: 0.0592
Epoch 7/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 83s 4ms/step - accuracy: 0.9808 - loss: 0.0594
Epoch 8/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 83s 4ms/step - accuracy: 0.9803 - loss: 0.0613
Epoch 9/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 83s 4ms/step - accuracy: 0.9805 - loss: 0.0620
Epoch 10/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 146s 4ms/step - accuracy: 0.9802 - loss: 0.0635
Epoch 11/15
21442/21442 ━━━━━━━━━━━━━━━━━━━━ 82s 4ms/step - accuracy: 0.9800 - loss:

In [38]:
# Separating features (X) and target (y)
X_test = test_data.drop(columns=['Label'])
y_test = test_data['Label']

# Standardizing features
scaler1 = StandardScaler()
X_test_scaled = scaler1.fit_transform(X_test)

# Applying K-Means SMOTE to handle class imbalance
# Use kmeans_estimator to specify the desired number of clusters
kmeans_smote1 = KMeansSMOTE(random_state=42, k_neighbors=2,
                           cluster_balance_threshold=0.1,
                           kmeans_estimator=KMeans(n_clusters=10))  # Specify n_clusters here
X_resampled, y_resampled = kmeans_smote1.fit_resample(X_test_scaled, y_test)

# Now `X_resampled` and `y_resampled` contain the oversampled data
# Convert resampled data back to a DataFrame
X_resampled_df = pd.DataFrame(X_resampled, columns=X_test.columns)
y_resampled_df = pd.DataFrame(y_resampled, columns=['Label'])

# Combine features and target into a single DataFrame
test_data = pd.concat([X_resampled_df, y_resampled_df], axis=1)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/_tags.py:354: FutureWarning: The KMeansSMOTE or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Prediction for Test Data
y_test_true = test_data['Label'].values
y_test_pred_prob = esn_dnn_model.predict(test_data.drop('Label', axis=1).values)
y_test_pred = np.argmax(y_test_pred_prob, axis=1)

# Prediction for Train Data
y_train_true = train_data['Label'].values
y_train_pred_prob = esn_dnn_model.predict(train_data.drop('Label', axis=1).values)
y_train_pred = np.argmax(y_train_pred_prob, axis=1)

18373/18373 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step
42883/42883 ━━━━━━━━━━━━━━━━━━━━ 81s 2ms/step


In [39]:


def evaluate_metrics(y_true, y_pred, average='macro'):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=average),
        "F1 Score": f1_score(y_true, y_pred, average=average),


    }
    return metrics


# Evaluate for Test Data
test_metrics = evaluate_metrics(y_test_true, y_test_pred)
print("Test Data Metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")

# Evaluate for Train Data
train_metrics = evaluate_metrics(y_train_true, y_train_pred)
print("\nTrain Data Metrics:")
for key, value in train_metrics.items():
    print(f"{key}: {value:.4f}")

# Classification Report
print("\nClassification Report for Test Data:\n")
print(classification_report(y_test_true, y_test_pred))






Test Data Metrics:
Accuracy: 0.9283
Precision: 0.9357
Recall: 0.9283
F1 Score: 0.9280

Train Data Metrics:
Accuracy: 0.9837
Precision: 0.9839
Recall: 0.9837
F1 Score: 0.9837

Classification Report for Test Data:

              precision    recall  f1-score   support

           0       0.88      0.99      0.93    293962
           1       0.99      0.86      0.92    293964

    accuracy                           0.93    587926
   macro avg       0.94      0.93      0.93    587926
weighted avg       0.94      0.93      0.93    587926



In [ ]:
y_test_pred_prob

In [22]:
y_train_pred

array([0, 0, 0, ..., 1, 1, 1])

In [23]:
y_test_true

array([0, 0, 0, ..., 1, 1, 1])

array([0, 0, 0, ..., 1, 1, 1])